# Image Classification Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: A deterministic synthetic dataset

CIFAR-10 lives on disk. To make this lesson reproducible and fast we build a synthetic dataset that looks like CIFAR — 32x32 RGB images with class-specific structure the model must learn. The exact same pipeline works unchanged on real CIFAR-10.

In [ ]:
```python

import numpy as np

import torch

from torch.utils.data import Dataset

def synthetic_cifar(num_per_class=1000, num_classes=10, seed=0):

    rng = np.random.default_rng(seed)

    X = []

    Y = []

    for c in range(num_classes):

        centre = rng.uniform(0, 1, (3,))

        freq = 2 + c

        for _ in range(num_per_class):

            yy, xx = np.meshgrid(np.linspace(0, 1, 32), np.linspace(0, 1, 32), indexing="ij")

            r = np.sin(xx * freq) * 0.5 + centre[0]

            g = np.cos(yy * freq) * 0.5 + centre[1]

            b = (xx + yy) * 0.5 * centre[2]

            img = np.stack([r, g, b], axis=-1)

            img += rng.normal(0, 0.08, img.shape)

            img = np.clip(img, 0, 1)

            X.append(img.astype(np.float32))

            Y.append(c)

    X = np.stack(X)

    Y = np.array(Y)

    idx = rng.permutation(len(X))

    return X[idx], Y[idx]

class ArrayDataset(Dataset):

    def __init__(self, X, Y, transform=None):

        self.X = X

        self.Y = Y

        self.transform = transform

    def __len__(self):

        return len(self.X)

    def __getitem__(self, i):

        img = self.X[i]

        if self.transform is not None:

            img = self.transform(img)

        img = torch.from_numpy(img).permute(2, 0, 1)

        return img, int(self.Y[i])

In [ ]:
```

Each class gets its own colour palette and frequency pattern, plus Gaussian noise to force the model to learn the signal rather than memorise pixels. Ten classes, one thousand images each, permuted.

### Step 2: Normalisation and augmentation

The two transforms that every vision pipeline has.

In [ ]:
```python

def standardize(mean, std):

    mean = np.array(mean, dtype=np.float32)

    std = np.array(std, dtype=np.float32)

    def _fn(img):

        return (img - mean) / std

    return _fn

def random_hflip(p=0.5):

    def _fn(img):

        if np.random.random() < p:

            return img[:, ::-1, :].copy()

        return img

    return _fn

def random_crop(pad=4):

    def _fn(img):

        h, w = img.shape[:2]

        padded = np.pad(img, ((pad, pad), (pad, pad), (0, 0)), mode="reflect")

        y = np.random.randint(0, 2 * pad)

        x = np.random.randint(0, 2 * pad)

        return padded[y:y + h, x:x + w, :]

    return _fn

def compose(*fns):

    def _fn(img):

        for fn in fns:

            img = fn(img)

        return img

    return _fn

In [ ]:
```

Reflect-pad before crop, not zero-pad, because black borders are a signal the model would learn to ignore in a non-useful way.

### Step 3: Mixup

Mixes two images and two labels inside the training step. Implemented as a batch transform so it lives next to the forward pass rather than inside the dataset.

In [ ]:
```python

def mixup_batch(x, y, num_classes, alpha=0.2):

    if alpha <= 0:

        return x, torch.nn.functional.one_hot(y, num_classes).float()

    lam = float(np.random.beta(alpha, alpha))

    idx = torch.randperm(x.size(0), device=x.device)

    x_mixed = lam * x + (1 - lam) * x[idx]

    y_onehot = torch.nn.functional.one_hot(y, num_classes).float()

    y_mixed = lam * y_onehot + (1 - lam) * y_onehot[idx]

    return x_mixed, y_mixed

def soft_cross_entropy(logits, soft_targets):

    log_probs = torch.log_softmax(logits, dim=-1)

    return -(soft_targets * log_probs).sum(dim=-1).mean()

In [ ]:
```

`soft_cross_entropy` is cross-entropy against a soft-label distribution. It reduces to the usual one-hot case when the target is exactly one-hot.

### Step 4: The training loop

The complete recipe: one pass over the data, gradients once per batch, scheduler stepped once per epoch.

In [ ]:
```python

import torch

import torch.nn as nn

from torch.utils.data import DataLoader

from torch.optim import SGD

from torch.optim.lr_scheduler import CosineAnnealingLR

def train_one_epoch(model, loader, optimizer, device, num_classes, use_mixup=True):

    model.train()

    total, correct, loss_sum = 0, 0, 0.0

    for x, y in loader:

        x, y = x.to(device), y.to(device)

        if use_mixup:

            x_m, y_soft = mixup_batch(x, y, num_classes)

            logits = model(x_m)

            loss = soft_cross_entropy(logits, y_soft)

        else:

            logits = model(x)

            loss = nn.functional.cross_entropy(logits, y, label_smoothing=0.1)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        loss_sum += loss.item() * x.size(0)

        total += x.size(0)

        # Training accuracy vs the un-mixed labels `y` is only an approximation

        # when mixup is on (the model saw soft targets, not y). Treat it as a

        # rough progress signal; rely on val accuracy for real performance.

        with torch.no_grad():

            pred = logits.argmax(dim=-1)

            correct += (pred == y).sum().item()

    return loss_sum / total, correct / total

@torch.no_grad()

def evaluate(model, loader, device, num_classes):

    model.eval()

    total, correct = 0, 0

    loss_sum = 0.0

    cm = torch.zeros(num_classes, num_classes, dtype=torch.long)

    for x, y in loader:

        x, y = x.to(device), y.to(device)

        logits = model(x)

        loss = nn.functional.cross_entropy(logits, y)

        pred = logits.argmax(dim=-1)

        for t, p in zip(y.cpu(), pred.cpu()):

            cm[t, p] += 1

        loss_sum += loss.item() * x.size(0)

        total += x.size(0)

        correct += (pred == y).sum().item()

    return loss_sum / total, correct / total, cm

In [ ]:
```

Five invariants you check every time you write a training loop:

1. `model.train()` before training, `model.eval()` before evaluation — flips dropout and batchnorm behaviour.

2. `.zero_grad()` before `.backward()`.

3. `.item()` when accumulating metrics so nothing keeps the computation graph alive.

4. `@torch.no_grad()` during evaluation — saves memory and time, prevents subtle accidents.

5. Argmax against raw logits, not softmax — same result, one fewer op.

### Step 5: Put it together

Use the `TinyResNet` from the previous lesson, train for a few epochs, evaluate.

In [ ]:
```python

from main import synthetic_cifar, ArrayDataset

from main import standardize, random_hflip, random_crop, compose

from main import mixup_batch, soft_cross_entropy

from main import train_one_epoch, evaluate

# TinyResNet comes from the previous lesson (03-cnns-lenet-to-resnet).

# Adjust the import path to wherever you stored the previous lesson's code.

from cnns_lenet_to_resnet import TinyResNet  # example placeholder

X, Y = synthetic_cifar(num_per_class=500)

split = int(0.9 * len(X))

X_train, Y_train = X[:split], Y[:split]

X_val, Y_val = X[split:], Y[split:]

mean = [0.5, 0.5, 0.5]

std = [0.25, 0.25, 0.25]

train_tf = compose(random_hflip(), random_crop(pad=4), standardize(mean, std))

eval_tf = standardize(mean, std)

train_ds = ArrayDataset(X_train, Y_train, transform=train_tf)

val_ds = ArrayDataset(X_val, Y_val, transform=eval_tf)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=0)

val_loader = DataLoader(val_ds, batch_size=256, shuffle=False, num_workers=0)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = TinyResNet(num_classes=10).to(device)

optimizer = SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4, nesterov=True)

scheduler = CosineAnnealingLR(optimizer, T_max=10)

for epoch in range(10):

    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, device, 10, use_mixup=True)

    va_loss, va_acc, _ = evaluate(model, val_loader, device, 10)

    scheduler.step()

    print(f"epoch {epoch:2d}  lr {scheduler.get_last_lr()[0]:.4f}  "

          f"train {tr_loss:.3f}/{tr_acc:.3f}  val {va_loss:.3f}/{va_acc:.3f}")

In [ ]:
```

On the synthetic dataset, this gets to near-perfect validation accuracy within five epochs, which is the point: the pipeline is correct, the model can learn what is learnable. Swap the dataset for real CIFAR-10 and the same loop trains to ~90% without changes.

### Step 6: Read the confusion matrix

Accuracy alone never tells you where the model is failing. The confusion matrix does.

In [ ]:
```python

def print_confusion(cm, labels=None):

    c = cm.shape[0]

    labels = labels or [str(i) for i in range(c)]

    print(f"{'':>6}" + "".join(f"{l:>5}" for l in labels))

    for i in range(c):

        row = cm[i].tolist()

        print(f"{labels[i]:>6}" + "".join(f"{v:>5}" for v in row))

    print()

    tp = cm.diag().float()

    fp = cm.sum(dim=0).float() - tp

    fn = cm.sum(dim=1).float() - tp

    prec = tp / (tp + fp).clamp_min(1)

    rec = tp / (tp + fn).clamp_min(1)

    f1 = 2 * prec * rec / (prec + rec).clamp_min(1e-9)

    for i in range(c):

        print(f"{labels[i]:>6}  prec {prec[i]:.3f}  rec {rec[i]:.3f}  f1 {f1[i]:.3f}")

_, _, cm = evaluate(model, val_loader, device, 10)

print_confusion(cm)

In [ ]:
```

Rows are true classes, columns are predictions. A cluster of off-diagonal counts between classes 3 and 5 means the model confuses those two and gives you a starting point for targeted data collection or a class-specific augmentation.

## Exercises

In [ ]:
1. **(Easy)** Train the same model with and without mixup for five epochs on the synthetic dataset. Plot train and val loss for both. Explain why train loss with mixup is higher yet val accuracy is similar or better.
2. **(Medium)** Implement Cutout — zero out a random 8x8 square in each training image — and run an ablation vs no augmentation, hflip+crop, hflip+crop+cutout, hflip+crop+mixup. Report val accuracy for each.
3. **(Hard)** Build a CIFAR-100 pipeline (100 classes, same input size) and reproduce a ResNet-34 training run to within 1% of published accuracy. Extras: sweep three learning rates and two weight decays, log to a local CSV, produce the final confusion-matrix-top-confusions table.